In [ ]:
# checks that all observations - draw-time average is near equal for all observations
# should return 0
def observation_ave_close(post1, post2,n):
    diff_ge_1 = []
    cnt = 0
    for i in range(n):
        p1 = float(post1.isel(p_obs=i).mean())
        p2 = float(post2["mu"].isel(p_obs=i).mean())
        diff= p1-p2
        if abs(diff) > .001:
            diff_ge_1.append((i,p1,p2,diff))
            cnt+=1
    print(cnt)
    return diff_ge_1, cnt

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 1. Calculate the expected value (mean probability) for EVERY patient
# (Replace 'post1' with whatever your PyMC native output variable is named)
py_patient_means = post1.mean(dim=["sample"]).values  # Native PyMC
nb_patient_means = post2["mu"].mean(dim=["chain", "draw"]).values  # Numba Engine

# 2. Plot them against each other
plt.figure(figsize=(8, 8))
plt.scatter(py_patient_means, nb_patient_means, alpha=0.4, color='blue', edgecolor='none')

# Draw the perfect 1:1 match line
max_val = max(np.max(py_patient_means), np.max(nb_patient_means))
plt.plot([0, max_val], [0, max_val], 'k--', lw=2, label="Perfect Match (y=x)")

plt.xlabel("PyMC Native Expected Probability")
plt.ylabel("Numba Engine Expected Probability")
plt.title("Per-Patient Verification: PyMC vs Numba")
plt.legend()
plt.grid(True)
plt.show()

# Calculate the mean absolute error across the population
mae = np.mean(np.abs(py_patient_means - nb_patient_means))
print(f"Mean Absolute Error across {len(py_patient_means)} patients: {mae:.6f}")

In [ ]:
import os
import psutil
import gc

def check_ram(label="Current"):
    # Get the OS-level memory footprint of this specific Jupyter kernel
    process = psutil.Process(os.getpid())
    mem_mb = process.memory_info().rss / (1024 * 1024)
    print(f"[{label}] RAM Usage: {mem_mb:,.2f} MB")

# Check the baseline with all_trees still sitting in memory
check_ram("BEFORE NUKE")

In [ ]:
print(f"Number of trees currently in RAM: {len(BSM.all_trees)}")

# 1. Clear the list
BSM.all_trees.clear()

# 2. Force the Python Garbage Collector to run immediately
gc.collect()

# Check the memory after the nuke
check_ram("AFTER NUKE")

In [ ]:
# Assuming you saved the outputs as `old_preds` and `new_preds`
print("--- OVERALL POSTERIOR MEAN ---")
print(f"Old PyMC-BART Mean: {float(post1.mean()):.6f}")
print(f"New Numba Engine Mean: {float(post2['mu'].mean()):.6f}")

print("\n--- PATIENT 0 EXPECTED VALUE ---")
print(f"Old PyMC-BART Pt 0: {float(post1.isel(p_obs=200).mean()):.6f}")
print(f"New Numba Engine Pt 0: {float(post2['mu'].isel(p_obs=200).mean()):.6f}")

In [ ]:
import os
import psutil

def check_local_mac_ram():
    """
    Captures the exact resident memory footprint of your Python script
    and all parallel MCMC chain workers running locally on macOS.
    """
    try:
        main_pid = os.getpid()
        main_process = psutil.Process(main_pid)
        
        # Grab memory of the main script/notebook process
        total_rss = main_process.memory_info().rss
        
        # Find all parallel worker processes spawned for the chains
        workers = main_process.children(recursive=True)
        
        for worker in workers:
            try:
                # RSS is the actual physical memory occupied in RAM
                total_rss += worker.memory_info().rss
            except (psutil.NoSuchProcess, psutil.AccessDenied):
                # Handles workers shutting down or cycling mid-check
                continue
                
        print(f"Total Local RAM Usage: {total_rss / (1024**3):.2f} GB")
        return total_rss
        
    except Exception as e:
        print(f"Error checking local macOS memory: {e}")
        return None

check_local_mac_ram()

In [ ]:
import numpy as np

def safe_pdp_predict(model, X_pdp, chunk_size=100000):
    """
    Slices a massive PDP dataset into safe chunks to prevent 
    instantaneous RAM spikes and kernel crashes on local machines.
    """
    n_rows = X_pdp.shape[0]
    all_preds = []
    
    # Loop through the dataset in safe increments
    for i in range(0, n_rows, chunk_size):
        X_chunk = X_pdp[i:i+chunk_size]
        
        # Use your fast prediction method with the offset included
        chunk_preds = predict_with_offset(model, X_chunk) 
        
        # If your prediction function returns full posterior draws (rows x draws),
        # take the mean across draws right here to collapse the memory down instantly!
        chunk_mean = np.mean(chunk_preds, axis=1) 
        
        all_preds.append(chunk_mean)
        
    # Concatenate the safely collapsed 1D arrays back together
    return np.concatenate(all_preds)